In [13]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, r2_score
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import numpy as np
import seaborn as sns
from sklearn import preprocessing
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from ucimlrepo import fetch_ucirepo 


xve = ''
yve = 'LeaveOrNot'
fpv = 'datasets/Employee.csv'

#imports data
origData = pd.read_csv(fpv)
df = origData
#del df['full_name']
#displays first 5 rows
print(df.head())

#checks for missing values
print(df.isnull().any())

#fills missing values
df.fillna(df.mean(numeric_only=True).round(1), inplace=True)
string_columns = df.select_dtypes(include=['object']).columns
df[string_columns] = df[string_columns].fillna(df[string_columns].mode().iloc[0])
#print(df.head())
#print(df.isnull().any())

#oneHotEncoding
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))
df_encoded = pd.concat([df, one_hot_df], axis=1)
df_encoded = df_encoded.drop(categorical_columns, axis=1)
print(df_encoded.head())

#getting columns in x
abc = df_encoded.columns.get_loc(yve)
test2 = []
test3 = []
for i in range(len(df.columns)):
    if(i != abc):
        test2.append(i)
limit = 5
for i in range(len(test2)):
    if(i<limit):
        test3.append(df_encoded.columns[test2[i]])


#sets x and y
X = df_encoded[test3]
y = df_encoded[yve]


   Education  JoiningYear       City  PaymentTier  Age  Gender EverBenched  \
0  Bachelors         2017  Bangalore            3   34    Male          No   
1  Bachelors         2013       Pune            1   28  Female          No   
2  Bachelors         2014  New Delhi            3   38  Female          No   
3    Masters         2016  Bangalore            3   27    Male          No   
4    Masters         2017       Pune            3   24    Male         Yes   

   ExperienceInCurrentDomain  LeaveOrNot  
0                          0           0  
1                          3           1  
2                          2           0  
3                          5           1  
4                          2           1  
Education                    False
JoiningYear                  False
City                         False
PaymentTier                  False
Age                          False
Gender                       False
EverBenched                  False
ExperienceInCurrentDomain   

In [14]:
#80/20 train test split with random seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#first model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
#printing scores/evaulation
print(f'Accuracy: {accuracy}')
print(f'F1: {f1}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print('Classification Report:')
print(classification_report(y_test, y_pred))

#second model
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
#printing scores/evaulation
print(f'Accuracy (Random Forest): {accuracy_rf}')
print(f'F1 (Random Forest): {f1_rf}')
print(f'Precision (Random Forest): {precision_rf}')
print(f'Recall (Random Forest): {recall_rf}')
print('Classification Report (Random Forest):')
print(classification_report(y_test, y_pred_rf))


Accuracy: 0.640171858216971
F1: 0.2191142191142191
Precision: 0.4351851851851852
Recall: 0.14641744548286603
Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.90      0.77       610
           1       0.44      0.15      0.22       321

    accuracy                           0.64       931
   macro avg       0.55      0.52      0.49       931
weighted avg       0.59      0.64      0.58       931

Accuracy (Random Forest): 0.8055853920515574
F1 (Random Forest): 0.6905982905982906
Precision (Random Forest): 0.7651515151515151
Recall (Random Forest): 0.6292834890965732
Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       610
           1       0.77      0.63      0.69       321

    accuracy                           0.81       931
   macro avg       0.79      0.76      0.77       931
weighted avg       0.80      0.81      0.80       931



In [15]:
#k fold implementation
